# 2. Split Time-Series Data
Split time-series data for forecasting without temporal leakage.

In [1]:
"""Split time-series data for forecasting without temporal leakage.
",

Purpose
-------
This script implements the agreed forecasting split design:

1. Hold out the last part of the timeline as the final test set.
2. Use the earlier part as the development set.
3. Inside the development set, create sklearn TimeSeriesSplit folds.
4. Support both ``sliding`` and ``expanding`` windows.
5. Export compatibility aliases ``train/`` and ``val/`` from the last fold.

Default input is the v3 ML mart preprocessing parquet, not feature-engineered
data. This keeps the pipeline easy to explain as:

    v3_preprocessing -> split boundaries -> point-in-time feature engineering

Feature engineering can still use historical context from previous splits to
compute lag/rolling features as long as each row only uses timestamps before
the prediction timestamp.

Implementation detail: sklearn's ``TimeSeriesSplit`` is applied to sorted
unique timestamps, not raw rows. The selected timestamp windows are then mapped
back to all site rows. This prevents the same timestamp from being split across
train/validation for different solar sites.

No random split is used. Test is never part of any fold.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

import pandas as pd
from sklearn.model_selection import TimeSeriesSplit


PROJECT_ROOT = Path.cwd().resolve().parents[3]
DEFAULT_INPUT = PROJECT_ROOT / "data" / "mlmart_base" / "v3_final_cleaned.parquet"
DEFAULT_OUTPUT_DIR = PROJECT_ROOT / "data" / "model" / "v3"

SITE_COL = "site_id"
TIMESTAMP_COL = "timestamp"
TARGET_COL = "energy_generated_kwh"

SplitStrategy = Literal["sliding", "expanding"]


@dataclass(frozen=True)
class TimeSeriesSplitConfig:
    """Configuration for forecasting-safe time-series split."""

    input_path: Path = DEFAULT_INPUT
    output_dir: Path = DEFAULT_OUTPUT_DIR
    strategy: SplitStrategy = "sliding"
    test_ratio: float = 0.15
    n_splits: int = 5
    sliding_train_blocks: int = 3
    timestamp_col: str = TIMESTAMP_COL
    site_col: str = SITE_COL
    target_col: str = TARGET_COL
    version: str = "v3"


def require_columns(df: pd.DataFrame, columns: list[str] | tuple[str, ...]) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


def load_input(config: TimeSeriesSplitConfig) -> pd.DataFrame:
    if not config.input_path.exists():
        raise FileNotFoundError(f"Input parquet not found: {config.input_path}")

    df = pd.read_parquet(config.input_path)
    require_columns(df, [config.timestamp_col, config.site_col, config.target_col])
    df[config.timestamp_col] = pd.to_datetime(df[config.timestamp_col], errors="coerce")
    df = df[df[config.timestamp_col].notna()].copy()
    return df.sort_values([config.timestamp_col, config.site_col]).reset_index(drop=True)


def validate_config(config: TimeSeriesSplitConfig) -> None:
    if config.strategy not in ("sliding", "expanding"):
        raise ValueError(f"Unsupported strategy={config.strategy}")
    if not 0 < config.test_ratio < 1:
        raise ValueError(f"test_ratio must be between 0 and 1, got {config.test_ratio}")
    if config.n_splits < 2:
        raise ValueError(f"n_splits must be >= 2, got {config.n_splits}")
    if config.sliding_train_blocks < 1:
        raise ValueError(
            f"sliding_train_blocks must be >= 1, got {config.sliding_train_blocks}"
        )


def split_development_test(
    timestamps: pd.Series,
    *,
    test_ratio: float,
    n_splits: int,
) -> tuple[pd.Timestamp, pd.Series, pd.Series]:
    """Return test start timestamp, development timestamps and test timestamps."""

    if timestamps.empty:
        raise ValueError("No valid timestamps found")

    n_total = len(timestamps)
    test_start_idx = int(n_total * (1.0 - test_ratio))
    # Keep enough timestamps before test to construct folds.
    test_start_idx = min(max(test_start_idx, n_splits + 2), n_total - 1)

    test_start_ts = pd.Timestamp(timestamps.iloc[test_start_idx])
    development_ts = timestamps.iloc[:test_start_idx].reset_index(drop=True)
    test_ts = timestamps.iloc[test_start_idx:].reset_index(drop=True)
    return test_start_ts, development_ts, test_ts


def build_sklearn_time_series_folds(
    development_ts: pd.Series,
    *,
    strategy: SplitStrategy,
    n_splits: int,
    sliding_train_blocks: int,
) -> list[dict[str, object]]:
    """Build fold boundaries with sklearn TimeSeriesSplit on unique timestamps.

    Why this wrapper exists:
    - sklearn splits arrays by row position.
    - This dataset has many site rows for each timestamp.
    - Splitting raw rows can leak the same timestamp across train/validation.
    - Therefore we split the unique timestamp axis, then map each timestamp
      window back to all dataframe rows.

    Sliding uses ``max_train_size``; expanding leaves ``max_train_size=None``.
    """

    if len(development_ts) < n_splits + 2:
        raise ValueError(
            "Not enough development timestamps for TimeSeriesSplit: "
            f"timestamps={len(development_ts)}, n_splits={n_splits}"
        )

    timestamp_axis = pd.Series(development_ts).reset_index(drop=True)
    max_train_size: int | None = None

    if strategy == "sliding":
        total_blocks = n_splits + sliding_train_blocks
        test_size = len(timestamp_axis) // total_blocks
        max_train_size = test_size * sliding_train_blocks
    else:
        test_size = len(timestamp_axis) // (n_splits + 1)

    if test_size < 1:
        raise ValueError(
            "TimeSeriesSplit validation window would be empty: "
            f"timestamps={len(timestamp_axis)}, n_splits={n_splits}, strategy={strategy}"
        )
    if len(timestamp_axis) <= n_splits * test_size:
        raise ValueError(
            "Not enough timestamps before test for sklearn TimeSeriesSplit: "
            f"timestamps={len(timestamp_axis)}, n_splits={n_splits}, test_size={test_size}"
        )

    splitter = TimeSeriesSplit(
        n_splits=n_splits,
        test_size=test_size,
        max_train_size=max_train_size,
    )

    folds: list[dict[str, object]] = []
    for fold, (train_idx, val_idx) in enumerate(splitter.split(timestamp_axis), start=1):
        train_ts = timestamp_axis.iloc[train_idx].reset_index(drop=True)
        val_ts = timestamp_axis.iloc[val_idx].reset_index(drop=True)
        folds.append(
            {
                "fold": fold,
                "strategy": strategy,
                "splitter": "sklearn.model_selection.TimeSeriesSplit",
                "train_timestamp_count": int(len(train_ts)),
                "val_timestamp_count": int(len(val_ts)),
                "sklearn_test_size": int(test_size),
                "sklearn_max_train_size": max_train_size,
                "train_start_ts": pd.Timestamp(train_ts.iloc[0]),
                "train_end_ts": pd.Timestamp(train_ts.iloc[-1]),
                "val_start_ts": pd.Timestamp(val_ts.iloc[0]),
                "val_end_ts": pd.Timestamp(val_ts.iloc[-1]),
            }
        )
    return folds


def build_folds(
    development_ts: pd.Series,
    *,
    strategy: SplitStrategy,
    n_splits: int,
    sliding_train_blocks: int,
) -> list[dict[str, object]]:
    return build_sklearn_time_series_folds(
        development_ts,
        strategy=strategy,
        n_splits=n_splits,
        sliding_train_blocks=sliding_train_blocks,
    )


def filter_window(
    df: pd.DataFrame,
    *,
    timestamp_col: str,
    start_ts: pd.Timestamp,
    end_ts: pd.Timestamp,
) -> pd.DataFrame:
    mask = (df[timestamp_col] >= start_ts) & (df[timestamp_col] <= end_ts)
    return df.loc[mask].copy()


def summarize_part(
    *,
    name: str,
    part: pd.DataFrame,
    config: TimeSeriesSplitConfig,
    fold: int | None = None,
    role: str | None = None,
) -> dict[str, object]:
    row: dict[str, object] = {
        "name": name,
        "rows": int(len(part)),
        "site_count": int(part[config.site_col].nunique()) if config.site_col in part else 0,
        "min_timestamp": part[config.timestamp_col].min() if len(part) else pd.NaT,
        "max_timestamp": part[config.timestamp_col].max() if len(part) else pd.NaT,
        "target_null_rows": int(part[config.target_col].isna().sum())
        if config.target_col in part
        else 0,
    }
    if fold is not None:
        row["fold"] = fold
    if role is not None:
        row["role"] = role
    for col in (
        f"{config.version}_missing_weather_flag",
        f"{config.version}_outlier_flag",
        f"{config.version}_exclude_from_loss_flag",
        f"{config.version}_has_complete_history_features",
        f"{config.version}_gap_after_prev_flag",
    ):
        if col in part.columns:
            row[col] = int(part[col].fillna(False).sum())
    return row


def add_holdout_labels(
    df: pd.DataFrame,
    *,
    config: TimeSeriesSplitConfig,
    test_start_ts: pd.Timestamp,
) -> pd.DataFrame:
    out = df.copy()
    out[f"{config.version}_holdout_split"] = "development"
    out.loc[
        out[config.timestamp_col] >= test_start_ts,
        f"{config.version}_holdout_split",
    ] = "test"
    out[f"{config.version}_test_start_timestamp"] = test_start_ts
    out[f"{config.version}_split_strategy"] = config.strategy
    out[f"{config.version}_n_time_series_splits"] = config.n_splits
    return out


def write_readme(config: TimeSeriesSplitConfig, folds: list[dict[str, object]]) -> None:
    readme = config.output_dir / "README.md"
    final_fold = folds[-1]
    text = f"""# Time-series model split

This folder is generated by:

```bash
python srcs/05_machine_learning/03_forecasting_v3/01_split_time_series_data.py
```

## Design

- No random split.
- Final test is the last `{config.test_ratio:.0%}` of unique timestamps.
- Development set is all data before test.
- Time-series folds are generated inside development only.
- Fold engine: `sklearn.model_selection.TimeSeriesSplit`.
- Split axis: sorted unique `{config.timestamp_col}`, then mapped back to all site rows.
- Strategy: `{config.strategy}`.
- Number of folds: `{config.n_splits}`.
- Sliding train blocks: `{config.sliding_train_blocks}`.

## Output folders

- `development/{config.version}_development.parquet`: all pre-test data.
- `test/{config.version}_test.parquet`: final holdout test.
- `time_series_folds/fold_*_train.parquet`: fold train windows.
- `time_series_folds/fold_*_val.parquet`: fold validation windows.
- `train/{config.version}_train.parquet`: compatibility alias from final fold train.
- `val/{config.version}_val.parquet`: compatibility alias from final fold validation.
- `final_train/{config.version}_final_train.parquet`: same rows as development; use after model selection.
- `summaries/{config.version}_split_summary.csv`.
- `summaries/{config.version}_time_series_fold_summary.csv`.

## Final fold alias

- train: `{final_fold['train_start_ts']}` -> `{final_fold['train_end_ts']}`
- val: `{final_fold['val_start_ts']}` -> `{final_fold['val_end_ts']}`

## Point-in-time feature engineering note

After split boundaries are defined, lag/rolling features should be generated
with a point-in-time rule. For timestamp `t`, target-derived features must use
`shift(1)` or an equivalent rule so only observations before `t` are visible.
Validation/test rows may use historical context from previous splits, but never
future observations.
"""
    readme.write_text(text, encoding="utf-8")


def run_split(config: TimeSeriesSplitConfig) -> dict[str, Path]:
    validate_config(config)
    config.output_dir.mkdir(parents=True, exist_ok=True)

    df = load_input(config)
    timestamps = pd.Series(df[config.timestamp_col].dropna().sort_values().unique())
    test_start_ts, development_ts, test_ts = split_development_test(
        timestamps,
        test_ratio=config.test_ratio,
        n_splits=config.n_splits,
    )
    folds = build_folds(
        development_ts,
        strategy=config.strategy,
        n_splits=config.n_splits,
        sliding_train_blocks=config.sliding_train_blocks,
    )

    labeled = add_holdout_labels(df, config=config, test_start_ts=test_start_ts)
    development = labeled[labeled[f"{config.version}_holdout_split"].eq("development")].copy()
    test = labeled[labeled[f"{config.version}_holdout_split"].eq("test")].copy()
    final_fold = folds[-1]
    train_alias = filter_window(
        labeled,
        timestamp_col=config.timestamp_col,
        start_ts=final_fold["train_start_ts"],
        end_ts=final_fold["train_end_ts"],
    )
    val_alias = filter_window(
        labeled,
        timestamp_col=config.timestamp_col,
        start_ts=final_fold["val_start_ts"],
        end_ts=final_fold["val_end_ts"],
    )

    # Compatibility column for old training scripts.
    train_alias[f"{config.version}_split"] = "train"
    val_alias[f"{config.version}_split"] = "val"
    test = test.copy()
    test[f"{config.version}_split"] = "test"

    paths = {
        "development": config.output_dir / "development" / f"{config.version}_development.parquet",
        "test": config.output_dir / "test" / f"{config.version}_test.parquet",
        "final_train": config.output_dir / "final_train" / f"{config.version}_final_train.parquet",
        "train_alias": config.output_dir / "train" / f"{config.version}_train.parquet",
        "val_alias": config.output_dir / "val" / f"{config.version}_val.parquet",
        "split_summary": config.output_dir / "summaries" / f"{config.version}_split_summary.csv",
        "fold_summary": config.output_dir
        / "summaries"
        / f"{config.version}_time_series_fold_summary.csv",
        "config": config.output_dir / "summaries" / f"{config.version}_split_config.json",
    }
    for path in paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)

    development.to_parquet(paths["development"], index=False)
    test.to_parquet(paths["test"], index=False)
    development.to_parquet(paths["final_train"], index=False)
    train_alias.to_parquet(paths["train_alias"], index=False)
    val_alias.to_parquet(paths["val_alias"], index=False)

    split_summary = pd.DataFrame(
        [
            summarize_part(name="development", part=development, config=config),
            summarize_part(name="train_alias_final_fold", part=train_alias, config=config),
            summarize_part(name="val_alias_final_fold", part=val_alias, config=config),
            summarize_part(name="test", part=test, config=config),
        ]
    )
    split_summary.to_csv(paths["split_summary"], index=False)

    fold_rows: list[dict[str, object]] = []
    folds_dir = config.output_dir / "time_series_folds"
    folds_dir.mkdir(parents=True, exist_ok=True)
    for fold_info in folds:
        fold = int(fold_info["fold"])
        fold_train = filter_window(
            labeled,
            timestamp_col=config.timestamp_col,
            start_ts=fold_info["train_start_ts"],
            end_ts=fold_info["train_end_ts"],
        )
        fold_val = filter_window(
            labeled,
            timestamp_col=config.timestamp_col,
            start_ts=fold_info["val_start_ts"],
            end_ts=fold_info["val_end_ts"],
        )
        fold_train[f"{config.version}_cv_fold"] = fold
        fold_train[f"{config.version}_cv_role"] = "train"
        fold_val[f"{config.version}_cv_fold"] = fold
        fold_val[f"{config.version}_cv_role"] = "val"
        fold_train.to_parquet(folds_dir / f"fold_{fold}_train.parquet", index=False)
        fold_val.to_parquet(folds_dir / f"fold_{fold}_val.parquet", index=False)
        fold_rows.append(
            summarize_part(
                name=f"fold_{fold}_train",
                part=fold_train,
                config=config,
                fold=fold,
                role="train",
            )
        )
        fold_rows.append(
            summarize_part(
                name=f"fold_{fold}_val",
                part=fold_val,
                config=config,
                fold=fold,
                role="val",
            )
        )

    fold_summary = pd.DataFrame(fold_rows)
    fold_summary.to_csv(paths["fold_summary"], index=False)
    paths["config"].write_text(
        json.dumps(
            {
                **asdict(config),
                "input_path": str(config.input_path),
                "output_dir": str(config.output_dir),
                "test_start_timestamp": str(test_start_ts),
                "development_timestamp_count": int(len(development_ts)),
                "test_timestamp_count": int(len(test_ts)),
            },
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    # Backward-compatible summary copies for older notebooks/scripts.
    split_summary.to_csv(config.output_dir / f"{config.version}_split_summary.csv", index=False)
    fold_summary.to_csv(
        config.output_dir / f"{config.version}_time_series_fold_summary.csv",
        index=False,
    )
    write_readme(config, folds)

    print("Time-series split completed")
    print(f"input       : {config.input_path}")
    print(f"output_dir  : {config.output_dir}")
    print(f"strategy    : {config.strategy}")
    print(f"test_start  : {test_start_ts}")
    print("\nSplit summary:")
    print(split_summary.to_string(index=False))
    print("\nFold summary:")
    print(fold_summary.to_string(index=False))
    return paths


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Create forecasting-safe time-series splits.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--strategy", choices=["sliding", "expanding"], default="sliding")
    parser.add_argument("--test-ratio", type=float, default=0.15)
    parser.add_argument("--n-splits", type=int, default=5)
    parser.add_argument("--sliding-train-blocks", type=int, default=3)
    parser.add_argument("--timestamp-col", default=TIMESTAMP_COL)
    parser.add_argument("--site-col", default=SITE_COL)
    parser.add_argument("--target-col", default=TARGET_COL)
    parser.add_argument("--version", default="v3")
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    run_split(
        TimeSeriesSplitConfig(
            input_path=args.input.expanduser().resolve(),
            output_dir=args.output_dir.expanduser().resolve(),
            strategy=args.strategy,
            test_ratio=args.test_ratio,
            n_splits=args.n_splits,
            sliding_train_blocks=args.sliding_train_blocks,
            timestamp_col=args.timestamp_col,
            site_col=args.site_col,
            target_col=args.target_col,
            version=args.version,
        )
    )
    return 0


if __name__ == "__main__":
    # raise SystemExit(main())
    pass
